# AMS Sketch — 2nd Frequency Moment

## Learning Objectives

1. **Define** the $k$-th frequency moment $F_k = \sum_i m_i^k$ and explain why $F_2$ matters
2. **Construct** the AMS random variable and prove $E[X] = F_2$
3. **Analyse** the variance and derive the median-of-means estimator
4. **Implement** the practical $\pm 1$ hash variant using 4-wise independent hashing
5. **Apply** AMS to detect network anomalies and measure stream skewness


## Problem Statement

### Frequency Moments

Given a stream over a universe of size $n$, let $m_i$ = count of element $i$. The **$k$-th frequency moment** is:

$$F_k = \sum_{i=1}^{n} m_i^k$$

| Moment | Name | Meaning |
|--------|------|---------|
| $F_0$ | Distinct count | Number of distinct elements |
| $F_1$ | Stream length | Total elements (trivial: count them) |
| $F_2$ | **Surprise number** | Measure of concentration / skewness |
| $F_\infty$ | Max frequency | Count of most common element |

### Why $F_2$?

$F_2$ is small when the distribution is uniform ($F_2 = n$ if $m_i = 1$ for all $i$) and large when one element dominates ($F_2 \approx m_{\text{max}}^2$). In network monitoring, a sudden spike in $F_2$ indicates a traffic anomaly (e.g. a DDoS attack).

### The Challenge

Computing $F_2$ exactly requires storing all $m_i$ counts — $O(n)$ space. For $n = 10^9$ IP addresses this is gigabytes. AMS gives a $(1\pm\epsilon)$-approximation in $O(1/\epsilon^2 \cdot \log(1/\delta))$ space.


In [ ]:
from IPython.display import SVG, display
svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="360" font-family="monospace" font-size="12">
  <rect width="820" height="360" fill="#fafafa" rx="8"/>
  <text x="410" y="22" text-anchor="middle" fill="#333" font-size="13" font-weight="bold">AMS Sketch: Estimating 2nd Frequency Moment F₂</text>

  <!-- F2 definition -->
  <rect x="20" y="35" width="780" height="44" rx="4" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1.5"/>
  <text x="410" y="55" text-anchor="middle" fill="#4e79a7" font-size="12" font-weight="bold">F₂ = Σᵢ mᵢ²  (surprise number — measures concentration)</text>
  <text x="410" y="72" text-anchor="middle" fill="#555" font-size="11">where mᵢ = count of element i in the stream. F₂ is small if distribution is uniform, large if skewed.</text>

  <!-- Random variable X construction -->
  <rect x="20" y="90" width="380" height="165" rx="4" fill="#fff4e0" stroke="#f28e2b" stroke-width="1.5"/>
  <text x="210" y="110" text-anchor="middle" fill="#f28e2b" font-size="12" font-weight="bold">AMS Random Variable X</text>
  <text x="210" y="130" text-anchor="middle" fill="#333" font-size="11">Pick element a uniformly at random from stream</text>
  <text x="210" y="148" text-anchor="middle" fill="#333" font-size="11">Let X = (stream length) × (count of a after pick)</text>
  <text x="210" y="168" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">E[X] = F₂</text>
  <text x="210" y="190" text-anchor="middle" fill="#555" font-size="11">Proof sketch:</text>
  <text x="210" y="208" text-anchor="middle" fill="#555" font-size="11">P[pick a] × n × mₐ = (mₐ/n) × n × mₐ = mₐ²</text>
  <text x="210" y="226" text-anchor="middle" fill="#555" font-size="11">summing over all a → F₂</text>
  <text x="210" y="246" text-anchor="middle" fill="#555" font-size="10">(using "pick" = sample one stream position uniformly)</text>

  <!-- Variance reduction -->
  <rect x="420" y="90" width="380" height="165" rx="4" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="610" y="110" text-anchor="middle" fill="#59a14f" font-size="12" font-weight="bold">Variance Reduction (Chebyshev + Chernoff)</text>
  <text x="610" y="130" text-anchor="middle" fill="#333" font-size="11">Var[X] can be large — single estimate unreliable</text>
  <text x="610" y="150" text-anchor="middle" fill="#333" font-size="11">Strategy:</text>
  <text x="610" y="170" text-anchor="middle" fill="#333" font-size="11">① Run k independent copies → X₁,…,Xₖ</text>
  <text x="610" y="190" text-anchor="middle" fill="#333" font-size="11">② Average: Ȳ = (X₁+…+Xₖ)/k → reduces variance k×</text>
  <text x="610" y="212" text-anchor="middle" fill="#333" font-size="11">③ Run t groups of k; take median of group means</text>
  <text x="610" y="232" text-anchor="middle" fill="#555" font-size="10">Chebyshev bounds variance; Chernoff bounds median failure</text>
  <text x="610" y="248" text-anchor="middle" fill="#333" font-size="11">Space: O(k × t) = O((1/ε²) log(1/δ))</text>

  <!-- 4-wise independent hash trick -->
  <rect x="20" y="270" width="780" height="80" rx="5" fill="#f5f5f5" stroke="#ccc" stroke-width="1.5"/>
  <text x="410" y="290" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Practical: ±1 Random Variables via 4-wise Independent Hash</text>
  <text x="410" y="310" text-anchor="middle" fill="#333" font-size="11">Replace "sample stream position" with: maintain Z = Σ r(aᵢ) where r: elements → {-1,+1}</text>
  <text x="410" y="330" text-anchor="middle" fill="#333" font-size="11">r is a 4-wise independent hash function. Then X = Z² is an unbiased estimator of F₂.</text>
  <text x="410" y="348" text-anchor="middle" fill="#555" font-size="10">Proof uses 4-wise independence to show E[Z²] = Σmᵢ² = F₂ and bound variance.</text>
</svg>
'''
display(SVG(svg))


## Derivation

### The AMS Random Variable

Pick a stream position $p$ uniformly at random from $\{1, \ldots, |\text{stream}|\}$. Let $a$ be the element at position $p$. Define:

$$X = |\text{stream}| \times (\text{count of } a \text{ at positions } \geq p)$$

**Claim:** $E[X] = F_2$.

**Proof:**
$$E[X] = \sum_{i} \sum_{\text{positions } p \text{ of } i} \frac{1}{|\text{stream}|} \times |\text{stream}| \times (\text{count of } i \text{ from } p \text{ onward})$$

$$= \sum_i \sum_{j=1}^{m_i} j = \sum_i \frac{m_i(m_i+1)}{2} \approx \frac{F_2}{2}$$

(The exact form gives $F_2/2 + F_1/2$; we adjust for the constant.)

### Practical: $\pm 1$ Hash Variant

Maintain $Z = \sum_{\text{stream}} r(a)$ where $r: U \to \{-1, +1\}$ is a 4-wise independent hash. Then:

$$E[Z^2] = \sum_i m_i^2 + \sum_{i \neq j} m_i m_j E[r(i)] E[r(j)] = F_2$$

Since $E[r(i)] = 0$ for any 2-wise independent hash, all cross terms vanish. 4-wise independence is needed to bound the **variance** of $Z^2$.

### Space Complexity

$k \times t$ counters where:
- $k = O(1/\epsilon^2)$ — controls error $\epsilon$ (via CLT)
- $t = O(\log(1/\delta))$ — controls failure probability $\delta$ (via Chernoff)


## Algorithm Steps

1. **Choose** $k \times t$ independent hash functions $r_{ij}: U \to \{-1, +1\}$
2. **Initialise** counters $Z_{ij} = 0$
3. **For each stream element** $x$: for all $i, j$: $Z_{ij} \mathrel{+}= r_{ij}(x)$
4. **Within each group** $i$: compute mean of $Z_{i,1}^2, \ldots, Z_{i,k}^2$
5. **Return** median of the $t$ group means as $\hat{F}_2$


In [ ]:
import numpy as np


class AMSSketch:
    """
    AMS Sketch for estimating the 2nd frequency moment F₂ = Σᵢ mᵢ².

    Uses the ±1 random variable variant: maintains k×t counters Z,
    each updated as Z += r(element) where r: elements → {-1, +1}
    is a 4-wise independent hash function.

    Inputs
    ------
    k     : int — counters per group (controls variance: error ∝ 1/√k)
    t     : int — number of groups (controls failure probability)
    """

    def __init__(self, k=50, t=10):
        self.k = k
        self.t = t
        rng = np.random.default_rng(7)
        p = 2_147_483_647  # Mersenne prime
        # Each of the k*t sketches has independent hash params
        self.a = rng.integers(1, p, size=(t, k))
        self.b = rng.integers(0, p, size=(t, k))
        self.Z = np.zeros((t, k), dtype=float)  # counters

    def _r(self, x):
        """Map element x to ±1 per (t, k) hash functions via 4-wise independence trick."""
        # Approximate 4-wise independence using h(x) mod 2
        hv = (self.a * (hash(x) % 2_147_483_647) + self.b) % 2_147_483_647
        # Map even → +1, odd → -1
        return np.where(hv % 2 == 0, 1.0, -1.0)

    def update(self, element, count=1):
        """Process one stream element (or add 'count' copies)."""
        r_vals = self._r(element)  # shape (t, k)
        self.Z += count * r_vals

    def estimate(self):
        """
        Estimate F₂ = Σᵢ mᵢ².
        Each counter Z[i,j] estimates F₂ as Z[i,j]².
        Average within each group, then take median across groups.
        """
        estimates_per_group = np.mean(self.Z ** 2, axis=1)  # shape (t,)
        return float(np.median(estimates_per_group))


# ── Demo ──────────────────────────────────────────────────────────────────────
rng = np.random.default_rng(0)
n = 10_000

# Uniform stream: F₂ ≈ n × (1/d)² × d = n/d  (each of d items appears n/d times)
for d in [100, 1000]:
    counts = {i: n // d for i in range(d)}
    true_f2 = sum(c**2 for c in counts.values())

    sketch = AMSSketch(k=100, t=20)
    for item, cnt in counts.items():
        sketch.update(item, count=cnt)

    est = sketch.estimate()
    print(f"d={d:>5}  true F₂={true_f2:>10.0f}  estimate={est:>10.0f}  error={abs(est-true_f2)/true_f2*100:.1f}%")

# Skewed stream (Zipf): F₂ much larger
print("\nSkewed (Zipf) stream:")
frequencies = {i: int(10000 / (i + 1)) for i in range(100)}
true_f2 = sum(c**2 for c in frequencies.values())
sketch = AMSSketch(k=200, t=30)
for item, cnt in frequencies.items():
    sketch.update(item, count=cnt)
est = sketch.estimate()
print(f"true F₂={true_f2:>10.0f}  estimate={est:>10.0f}  error={abs(est-true_f2)/true_f2*100:.1f}%")
